<div align="center">

# CLIPPR

### Design a PPR protein that binds any RNA sequence you choose

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SyedZainAliShah/clippr/blob/main/notebooks/CLIPPR_designer.ipynb)
[![License: MIT](https://img.shields.io/badge/License-MIT-1a7f5a.svg)](https://github.com/SyedZainAliShah/clippr/blob/main/LICENSE)
[![Tests](https://img.shields.io/badge/tests-283%20passing-1a7f5a.svg)](https://github.com/SyedZainAliShah/clippr)

**iGEM Marburg 2026**

</div>

<div align="center">

<img src="https://raw.githubusercontent.com/SyedZainAliShah/clippr/main/notebooks/pipeline.svg" alt="CLIPPR pipeline: a target RNA passes through ppr, arelf, overhangs, codons, assembly and qc to become orderable DNA fragments" width="100%" style="max-width:980px">

</div>

---

Pentatricopeptide repeat proteins are built from tandem ~31-residue repeats, and **each
repeat reads exactly one RNA base** through two specificity residues:

| 5th + last residue | reads |     | 5th + last residue | reads |
|:---:|:---:|---|:---:|:---:|
| `T` `N` | **A** |  | `T` `D` | **G** |
| `N` `N` | **C** |  | `N` `D` | **U** |

So the protein is a deterministic function of your target — no catalogue to search. Give it
nine bases and you get a nine-repeat protein, a synthesisable coding sequence, a Golden Gate
assembly plan, and the fragments to order.

**Run the cells top to bottom.** Only the *Design parameters* cell normally needs editing.

> ##### Before you read any number
> **Predicted fidelity** comes from published ligation-count matrices (Pryor *et al.* 2020) —
> it is not a measured assembly efficiency in your hands. **QC** is a sequence-complexity
> check, not calibrated against vendor outcomes. **Cost** is a list price, not a quote.
> Nothing here has been validated at the bench.

In [ ]:
#@title Setup — install CLIPPR {display-mode: "form"}
#@markdown Installs the package if it is not already available. Safe to re-run — it never
#@markdown reinstalls over a working copy.
REPO = "SyedZainAliShah/clippr"

try:
    import clippr
    _msg = f"clippr {clippr.__version__} already available"
except ImportError:
    token = None
    try:
        from google.colab import userdata          # private-repo fallback
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        pass
    url = (f"git+https://{token}@github.com/{REPO}.git" if token
           else f"git+https://github.com/{REPO}.git")
    %pip install --quiet $url
    import clippr
    _msg = f"installed clippr {clippr.__version__}"

from IPython.display import HTML, display
display(HTML(
    f'<div style="border-left:3px solid #1a7f5a;padding:.5em .9em;'
    f'font-family:ui-monospace,monospace;font-size:13px;opacity:.85">{_msg}</div>'))

In [ ]:
#@title Design parameters — edit these {display-mode: "form"}
#@markdown ### Target
#@markdown The RNA sequence the PPR should bind. **Its length sets the architecture** —
#@markdown 9, 14 or 19 bases give a 9S, 14S or 19S protein.
target_rna = "AAAAUGUGG"  #@param {type:"string"}

#@markdown ### Host
#@markdown The genetic code follows automatically — nuclear hosts use table 1, chloroplasts
#@markdown table 11. **The two Chlamydomonas entries are not interchangeable:** the nucleus
#@markdown is GC-rich and prefers Leu `CTG` (0.73); the chloroplast is AT-rich and prefers
#@markdown Leu `TTA` (0.74). Using one for the other produces DNA that looks fine and is wrong.
organism = "c_reinhardtii_nuclear"  #@param ["c_reinhardtii_nuclear", "c_reinhardtii_chloroplast", "e_coli", "s_cerevisiae", "a_thaliana_nuclear", "n_tabacum_chloroplast"]

#@markdown ### Or bring your own codon usage
#@markdown Three ways, in order of precedence. Leave all blank to use the host above.
#@markdown
#@markdown **A file** — a `codon,frequency` CSV or a CDS FASTA. Upload it with the folder
#@markdown icon in the sidebar, or run the upload cell below, then put the filename here.
codon_table_file = ""  #@param {type:"string"}
#@markdown **A Kazusa species ID** — any NCBI taxonomy id Kazusa carries, e.g. `4577` for
#@markdown maize. Fetched and cached on first use.
kazusa_taxid = 0  #@param {type:"integer"}
#@markdown **The genetic code** to go with a table you supplied. Leave 0 to inherit from
#@markdown the host. Set 11 for anything organellar — a nuclear code on a chloroplast
#@markdown construct produces DNA that looks fine and is wrong.
genetic_code_override = 0  #@param {type:"integer"}

#@markdown ### Which enzyme sites must be absent
#@markdown `assembly` — this assembly's own chemistry (BsaI, BbsI)
#@markdown &nbsp;&nbsp;·&nbsp; `igem_rfc1000` — adds SapI, required by iGEM's Type IIS standard
#@markdown &nbsp;&nbsp;·&nbsp; `moclo_compat` — adds BsmBI to keep later MoClo levels open,
#@markdown a preference that can make some junctions infeasible
enzyme_profile = "igem_rfc1000"  #@param ["assembly", "igem_rfc1000", "moclo_compat"]

#@markdown **Extra sites to keep clear** — anything else this particular experiment needs
#@markdown absent, beyond the profile. Comma-separated, any name Biopython knows.
#@markdown Examples: `EcoRI, BamHI, HindIII, NotI`.
extra_blacklist = ""  #@param {type:"string"}

#@markdown ### Assembly
#@markdown Which Type IIS enzyme cuts the fragments out, and which published mis-ligation
#@markdown table scores the junctions. `BsaI-HFv2` and `BbsI-HF` are the two measured in
#@markdown Pryor *et al.* 2020 at 25 °C over 18 h.
assembly_enzyme = "BsaI"  #@param ["BsaI", "BbsI", "BsmBI", "SapI"]
ligation_table = "BsaI-HFv2"  #@param ["BsaI-HFv2", "BbsI-HF"]

#@markdown ### Destination vector level
#@markdown Or type your own acceptor overhangs below as `5prime,3prime` coding sites —
#@markdown they override the level. Remember the 3' entry is the **coding site**; the
#@markdown enzyme leaves its reverse complement.
destination_level = "level0"  #@param ["level_minus1", "level0", "level1"]
custom_destination = ""  #@param {type:"string"}

#@markdown ### Fragments
#@markdown How many pieces to split the gene into. Leave at 0 to let the length decide —
#@markdown set it only if your vendor has an awkward limit.
n_fragments = 0  #@param {type:"integer"}

#@markdown ### Reproducibility
#@markdown The same seed always gives the same design.
seed = 42  #@param {type:"integer"}
write_files = True  #@param {type:"boolean"}
check_offtarget = True  #@param {type:"boolean"}

In [ ]:
#@title Upload a codon table (optional) {display-mode: "form"}
#@markdown Optional. Run this only if you want to upload a codon table from your computer
#@markdown rather than type a path. It puts the file in the runtime and fills in the
#@markdown filename for you — then re-run the Design cell.
try:
    from google.colab import files as _f
    _up = _f.upload()
    if _up:
        codon_table_file = list(_up)[0]
        print(f"using {codon_table_file}")
except ImportError:
    print("not running on Colab — put the file beside the notebook and give its path "
          "in codon_table_file instead.")

In [ ]:
#@title Design — run this {display-mode: "form"}
#@markdown Picks cut positions and Golden Gate overhangs **first**, then codon-optimises with
#@markdown those positions locked — optimising first would let the optimiser rewrite the very
#@markdown bases the junctions depend on.
from clippr import DESTINATION_OVERHANGS, design_oneshot, table_from_kazusa
from IPython.display import HTML, display

result = design_oneshot(
    target_rna,
    organism=organism,
    codon_table=(codon_table_file or
                 (table_from_kazusa(kazusa_taxid) if kazusa_taxid else None)),
    genetic_code=genetic_code_override or None,
    enzyme_profile=enzyme_profile,
    extra_blacklist=extra_blacklist,
    enzyme=assembly_enzyme,
    matrix=ligation_table,
    destination=(tuple(s.strip().upper() for s in custom_destination.split(",")[:2])
                 if custom_destination else DESTINATION_OVERHANGS[destination_level]),
    n_fragments=n_fragments or None,
    seed=seed,
    check_offtarget=check_offtarget,
    outdir="clippr_output" if write_files else None,
)

qc = result["qc"]
TONE = {"PASS": "#1a7f5a", "WARNING": "#9a6b1f", "FAIL": "#a8402c"}
tone = TONE.get(qc["status"], "#6b7b75")


def _stat(label, value, hint=""):
    return (
        f'<div style="padding:.55em .9em .6em;border-left:1px solid rgba(128,145,138,.35)">'
        f'<div style="font-size:10.5px;letter-spacing:.09em;text-transform:uppercase;'
        f'opacity:.6">{label}</div>'
        f'<div style="font-size:19px;font-weight:600;font-variant-numeric:tabular-nums;'
        f'margin-top:.15em">{value}</div>'
        f'<div style="font-size:11.5px;opacity:.6">{hint}</div></div>')


ppr_code = result["ppr_code"]
if len(ppr_code) > 24:
    ppr_code = ppr_code[:24] + "…"

cards = "".join([
    _stat("architecture", result["architecture"], f'{len(result["protein"])} aa protein'),
    _stat("coding sequence", f'{len(result["cds"])} nt', ppr_code),
    _stat("fragments", len(result["oligos"]),
          "cuts at " + ", ".join(str(c) for c in result["cuts"])),
    _stat("fidelity", f'{result["fidelity"]:.3f}', "predicted, not measured"),
    _stat("GC", f'{qc["gc_pct"]:.1f}%',
          f'windows {qc["gc_window_min"]:.0f}–{qc["gc_window_max"]:.0f}%'),
    _stat("repeats", f'{qc["repeated_kmer_fraction"]:.1%}',
          f'longest {qc["longest_repeat"]} nt'),
])

warn = "".join(
    f'<div style="border-left:3px solid {TONE["WARNING"]};padding:.5em .9em;'
    f'margin-top:.7em;font-size:13px;line-height:1.5">{w}</div>'
    for w in result["warnings"])

RULE = "1px solid rgba(128,145,138,.35)"
card = (
    f'<div style="font-family:ui-sans-serif,system-ui,sans-serif;border:{RULE};'
    f'border-radius:5px;overflow:hidden;max-width:920px">'
    f'<div style="display:flex;align-items:center;gap:.8em;padding:.7em 1em;'
    f'border-bottom:{RULE}">'
    f'<span style="font-family:ui-monospace,monospace;font-size:16px;font-weight:600">'
    f'{result["target_rna"]}</span>'
    f'<span style="background:{tone};color:#fff;font-size:11px;font-weight:700;'
    f'letter-spacing:.07em;padding:.2em .7em;border-radius:99px">{qc["status"]}</span>'
    f'<span style="margin-left:auto;font-size:12.5px;opacity:.65">'
    f'{result["cost"]["total_eur"]:.2f} EUR list price · not a quote</span></div>'
    f'<div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(140px,1fr))">'
    f'{cards}</div></div>{warn}')

display(HTML(card))

## The fragments to order

One row per orderable piece. `oh5` and `oh3` are the four-base Golden Gate overhangs that
join each fragment to its neighbours.

In [ ]:
#@title Fragment table {display-mode: "form"}
cols = {"fragment_id": "fragment", "assembly_order": "order", "aa_length": "residues",
        "oligo_length": "oligo nt", "oh5_coding_site_5to3": "oh5",
        "oh3_coding_site_5to3": "oh3"}
table = result["oligos"][list(cols)].rename(columns=cols)

table.style.hide(axis="index").set_properties(
    subset=["oh5", "oh3"], **{"font-family": "ui-monospace, monospace"}).set_table_styles([
        {"selector": "th", "props": [("text-align", "left"), ("font-size", "11px"),
                                     ("letter-spacing", ".07em"), ("text-transform", "uppercase"),
                                     ("opacity", ".65"), ("padding", ".4em .9em")]},
        {"selector": "td", "props": [("padding", ".4em .9em"),
                                     ("font-variant-numeric", "tabular-nums")]}])

## Why this design, and not another

Every junction records the overhangs it *could* have used and what became of each:

- **selected** — the one used
- **considered** — feasible, but another scored at least as well
- **rejected** — no synonymous codon arrangement could avoid an excluded enzyme site, so
  that junction is *impossible* under the active profile, not merely worse

`local realizations` counts the synonymous arrangements still available around a junction.
It is reported, never used to choose — but a junction with 2 is more fragile than one with
16, and that is worth seeing before you order.

**If a design looks surprising, read this rather than trusting it.**

In [ ]:
#@title Design audit {display-mode: "form"}
audit = result["audit"]
print(audit.report())

rejected = audit.rejected
print(f"\n{len(rejected)} candidate overhang(s) ruled out entirely under "
      f"profile '{audit.enzyme_profile}'")
for d in rejected[:8]:
    print(f"    {d.sequence}  cut {d.junction_cut:>4d}   {d.reason}")
if len(rejected) > 8:
    print(f"    … and {len(rejected) - 8} more")
if not rejected:
    print("    (every achievable overhang was usable at every junction)")

## Take the files

Four artefacts: the order CSV, the oligos as FASTA, the assembled gene, and an annotated
GenBank record — every PPR repeat labelled with the base it reads — that opens directly in
Benchling or SnapGene.

In [ ]:
#@title Download the design files {display-mode: "form"}
import os
from IPython.display import HTML, display

if not result["paths"]:
    display(HTML('<div style="opacity:.7">Set <code>write_files</code> to True in the '
                 'parameters cell and re-run.</div>'))
else:
    try:
        from google.colab import files as colab_files
    except ImportError:
        colab_files = None

    LABEL = {"oligo_csv": "Order sheet (CSV)", "oligo_fasta": "Oligos (FASTA)",
             "gene_fasta": "Assembled gene (FASTA)", "genbank": "Annotated GenBank"}
    rows = "".join(
        f'<tr><td style="padding:.35em .9em">{LABEL.get(k, k)}</td>'
        f'<td style="padding:.35em .9em;font-family:ui-monospace,monospace;font-size:12px;'
        f'opacity:.7">{os.path.basename(p)}</td>'
        f'<td style="padding:.35em .9em;text-align:right;font-variant-numeric:tabular-nums;'
        f'opacity:.7">{os.path.getsize(p):,} B</td></tr>'
        for k, p in result["paths"].items())
    display(HTML(f'<table style="font-family:ui-sans-serif,system-ui,sans-serif;'
                 f'font-size:13px;border-collapse:collapse">{rows}</table>'))

    if colab_files:
        for p in result["paths"].values():
            colab_files.download(p)

## Does this target also exist in the chloroplast?

A PPR cannot tell which copy of a sequence you meant. If your target also occurs in an
endogenous chloroplast transcript, the protein binds there too and stops being specific to
your construct.

**A PPR binds RNA, so only transcripts count.** A match in a non-transcribed region is not
an RNA off-target, and neither is a reverse-complement match in DNA — the transcript from
that locus carries the other sequence. The scan reports both tiers so you can tell them
apart.

The *Chlamydomonas* chloroplast is 203,828 bases and 34.5% GC, with 109 annotated
transcripts covering 43.5% of it. Over 200 random targets of each length:

| target length | in genomic DNA | **in a transcript** |
|---|---|---|
| 9 nt | 97 of 200 (48%) | **32 of 200 (16%)** |
| 14 nt | 0 | 0 |
| 19 nt | 0 | 0 |

A nine-base sequence is not rare enough in a 204 kb genome; a fourteen-base one is. If a
target comes back flagged in the transcript tier, lengthening it is the reliable fix.

Occurrence is a *necessary* condition for an off-target interaction, never a sufficient
one. This reports sequence, not affinity — no binding is predicted.

In [ ]:
#@title Off-target check — does the host already contain this sequence? {display-mode: "form"}
from clippr.offtarget import architecture_advice, load_genome, load_transcripts, report, scan

genome = load_genome()
transcripts = load_transcripts()
gc = 100 * (genome.count("G") + genome.count("C")) / len(genome)
print(f"host: Chlamydomonas reinhardtii chloroplast, {len(genome):,} bp, {gc:.1f}% GC")
print()

print("expected occurrences by chance, for an average target:")
for n, e in architecture_advice(genome).items():
    print(f"  {n:>2}-nt target : {e:8.3f}")

print()
print(f"{len(transcripts)} annotated transcripts, "
      f"{sum(len(x.sequence) for x in transcripts):,} nt "
      f"({100*sum(len(x.sequence) for x in transcripts)/len(genome):.1f}% of the genome)")
print()
print(report([scan(target_rna, genome, transcripts)]))

---

## Designing a whole library

For a set of regulators, what matters is **orthogonality**: PPRᵢ must bind UTRᵢ and not
UTRⱼ. The matrix below is the pairwise distance between targets — larger is better
separated. Targets that sit close together risk one PPR binding another's UTR.

> `orthogonal.py` is a **capability, not a validated result**. Every other part of this
> package is checked against a 200-design corpus; this one has unit tests only, because no
> ground truth for it exists.

In [ ]:
#@title Design the whole library {display-mode: "form"}
targets = "AAAAUGUGG, GCUAAAGAC, UUACACGUG"  #@param {type:"string"}

from clippr import design_library

target_list = [t.strip().upper() for t in targets.split(",") if t.strip()]
lib = design_library(target_list, codon_table=None, organism=organism,
                     enzyme_profile=enzyme_profile, seed=seed,
                     check_offtarget=check_offtarget,
                     outdir="clippr_library" if write_files else None,
                     on_progress=lambda i, n, t: print(f"  {i}/{n}  {t}", flush=True))

print()
print(lib.summary())
print()
print(lib.crosstalk())

In [ ]:
#@title Library QC table {display-mode: "form"}
lib.qc_table()